
# 練習問題: 並列化して台数効果を測る

## 目標

`#pragma omp parallel for` (Fortran では `!$omp parallel do` ... `!$omp end parallel do`) を1つ挿入するだけで, 独立な繰り返しからなる重いループを並列化し, スレッド数を増やすと性能 (GFLOPS) が上がる「台数効果」を測定できることを体験する.

## 課題

`measure_speedup.cpp` (または `measure_speedup.f90`) は, 各要素 `x[i]` を独立に重い計算で求め, 実行時間と GFLOPS を表示する.
現状は逐次 (並列化されていない) なので, スレッド数を増やしても速くならない.

コメント `TODO` の指示に従って **OpenMP の指示行を1つ追加** し, ループを並列化せよ.

- C++: 計算本体の `for` ループの直前に `#pragma omp parallel for` を1行加える.
- Fortran: 計算本体の `do` ループを `!$omp parallel do` と `!$omp end parallel do` で囲む.

それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore measure_speedup.cpp -o measure_speedup.exe

# Fortran
nvfortran -fast -mp=multicore measure_speedup.f90 -o measure_speedup.exe
```

`OMP_NUM_THREADS` を 1, 2, 4, ... と変えながら実行し, GFLOPS を比較せよ. スレッドをコアに固定するため `OMP_PROC_BIND=true` も付ける.

```
OMP_PROC_BIND=true OMP_NUM_THREADS=1 ./measure_speedup.exe
OMP_PROC_BIND=true OMP_NUM_THREADS=2 ./measure_speedup.exe
OMP_PROC_BIND=true OMP_NUM_THREADS=4 ./measure_speedup.exe
```

第1引数で要素数 `m`, 第2引数で1要素あたりの反復数 `n` を変えられる (例: `./measure_speedup.exe 64 10000000`).

## 期待される結果

並列化後は, スレッド数を増やすと実行時間が短くなり, GFLOPS が増える (理想的にはスレッド数に比例). 例:

```
OMP_NUM_THREADS=1 ->  約  X GFLOPS
OMP_NUM_THREADS=2 -> 約 2X GFLOPS
OMP_NUM_THREADS=4 -> 約 4X GFLOPS
```

スレッド数を増やしても性能が頭打ちになる点も観察せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ measure_speedup.cpp
#include <cassert>
#include <cstdio>
#include <cstdlib>
#include <omp.h>

/* x = ax + b をひたすら n 回繰り返す.
   (|a| < 1.0 なら c によらず, x = b / (1 - a) に収束).
   n 回 mul + add を行う (-> 2 n flops) */
double lin_rec(double a, double b, double c, long n) {
  double x = c;
  for (long j = 0; j < n; j++) {
    x = a * x + b;
  }
  return x;
}

int main(int argc, char ** argv) {
  long m = (1 < argc ? atoi(argv[1]) : 64);
  long n = (2 < argc ? atoi(argv[2]) : 10 * 1000 * 1000);
  double * x = (double *)calloc(sizeof(double), m);
  assert(x);
  printf("m = %ld, n = %ld\n", m, n);
  /* 計測開始 */
  double t0 = omp_get_wtime();
  /* 計算本体 */
  // TODO: 下の for ループの直前に #pragma omp parallel for を1行追加し, ループを並列化せよ.
  for (long i = 0; i < m; i++) {
    x[i] = lin_rec(0.99, i + 1, 1.0, n);
  }
  /* 計測終了 */
  double t1 = omp_get_wtime();
  double dt = t1 - t0;          /* sec */
  /* 答え表示 (x[i] = 100 * (i + 1) くらいのはず) */
  for (long i = 0; i < m; i++) {
    printf("x[%3ld] = %9.3f\n", i, x[i]);
  }
  double flops = 2. * (double)m * (double)n;
  printf("elapsed    : %7.3f  sec\n", dt);
  printf("flops      : %.2e\n", flops);
  printf("%.3f GFLOPS\n", flops / dt * 1e-9);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore measure_speedup.cpp -o measure_speedup_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./measure_speedup_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ measure_speedup.f90
module lin_rec_mod
contains
  ! x = ax + b をひたすら n 回繰り返す.
  ! (|a| < 1.0 なら c によらず, x = b / (1 - a) に収束).
  ! n 回 mul + add を行う (-> 2 n flops)
  function lin_rec(a, b, c, n) result(x)
    real(8), intent(in) :: a, b, c
    integer(8), intent(in) :: n
    real(8) :: x
    integer(8) :: j
    x = c
    do j = 1, n
       x = a * x + b
    end do
  end function lin_rec
end module lin_rec_mod

program measure_speedup
  use omp_lib
  use lin_rec_mod
  character(len=32) :: arg
  integer(8) :: m, n, i
  real(8), allocatable :: x(:)
  real(8) :: t0, t1, dt, flops
  m = 64
  n = 10 * 1000 * 1000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) m
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) n
  end if
  allocate(x(m))
  print "(a,i0,a,i0)", "m = ", m, ", n = ", n
  ! 計測開始
  t0 = omp_get_wtime()
  ! 計算本体
  ! TODO: 下の do ループを !$omp parallel do ... !$omp end parallel do で囲み, ループを並列化せよ.
  do i = 1, m
     x(i) = lin_rec(0.99d0, real(i, 8), 1.0d0, n)
  end do
  ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
  ! 計測終了
  t1 = omp_get_wtime()
  dt = t1 - t0                  ! sec
  ! 答え表示 (x(i) = 100 * i くらいのはず)
  do i = 1, m
     print "(a,i3,a,f9.3)", "x[", i, "] = ", x(i)
  end do
  flops = 2.d0 * real(m, 8) * real(n, 8)
  print "(a,f7.3,a)", "elapsed    : ", dt, "  sec"
  print "(a,es9.2)",  "flops      : ", flops
  print "(f7.3,a)", flops / dt * 1e-9, " GFLOPS"
end program measure_speedup

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore measure_speedup.f90 -o measure_speedup_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./measure_speedup_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:measure_speedup.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:measure_speedup.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:measure_speedup.cpp}